In [2]:
from dotenv import load_dotenv
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

# 파싱

In [3]:
from langchain_upstage import UpstageDocumentParseLoader
from langchain.schema import Document

In [6]:
PATH = "data/2025-1수강편람_250217_split.pdf"

In [7]:
loader = UpstageDocumentParseLoader(
    PATH,
    coordinates=False,
    ocr='force',
    output_format="markdown",
    base64_encoding=[],
    split="page"
)

In [8]:
pages = loader.load()

In [9]:
# 목차를 날렸으니 페이지 수 추가
for page in pages:
    page.metadata["page"] += 2


In [10]:
# 혹시 모를 불상사를 위해 pkl로 저장
import pickle

with open("2025_md.pkl", "wb") as f:
    pickle.dump(pages, f)


# 임베딩

In [11]:
from langchain_upstage import UpstageEmbeddings
from langchain_chroma import Chroma

In [12]:
embeddings = UpstageEmbeddings(model="embedding-passage")

In [13]:
vectorstore = Chroma.from_documents(
    pages,
    embedding=embeddings,
    persist_directory="2025-1-0217",
    collection_name="2025-1-0217"
)

In [14]:
retriever = vectorstore.as_retriever()

retriever.invoke("24학번 졸업요건")

[Document(id='2c20c7b9-87a3-4570-a868-6d6f6f1629df', metadata={'page': 41}, page_content=" 2024학년도 입학자 교과과정 # 1. 졸업 기준 | 구 분 | 구 분 | 구 분 | 이 수 과 목 | 이 수 과 목 | 주 요 사 항 |\n| --- | --- | --- | --- | --- | --- |\n| 교 양 | 공통필수 | 공통필수 | 8개 교과목 | 8개 교과목 | 세종인을위한진로설계, 세종인을위한전공탐색, 창업과기업가정신1, 문제해결을위한글쓰기와발표, 서양철학:쟁점과토론, 우주자연인간, 취창업과진로설계, 대학영어 |\n| 교 양 | 계 열 별 필 수 | 균형교양 | 자신의 소속계열과 다른 3개 영역에서 9학점 선택 이수 (학생자율선택) | 자신의 소속계열과 다른 3개 영역에서 9학점 선택 이수 (학생자율선택) | 자신의 소속계열과 다른 3개 영역에서 9학점 선택 이수 (학생자율선택) |\n| 교 양 | 계 열 별 필 수 | 학문기초 교양 | 단과대학 또는 학과에 따라 지정된 과목 이수 | 단과대학 또는 학과에 따라 지정된 과목 이수 | 단과대학 또는 학과에 따라 지정된 과목 이수 |\n| 전공 | 전공 | 전공 | 구 분 | 내 용 | 내 용 |\n| 전공 | 전공 | 전공 | 단일전공 이수시 | 학과 또는 전공에 따라 차이가 있으므로 6항 확인 | 학과 또는 전공에 따라 차이가 있으므로 6항 확인 |\n| 전공 | 전공 | 전공 | 복수전공 이수시 (연계·융합 전공포함) | -전필 : 15 학점 -전선 : 24 학점 -합계 : 39 학점(주전공, 복수전공 각각이수) ※ 건축학전공 이수자는 11.전공, 복수.부.제2전공, 연계·융합전공, 세종인재자기 설계전공 신청 및 이수 안내 참조 ※ 교직과정 이수자가 교직복수전공시 주, 복수전공 각각 50학점 이상 이수 ※ 법학부, 호텔외식관광프랜차이즈경영학과와 국방시스템공학과, 항공 시스템공학과 등 계약학과의 복수전공에 관한 사항은 별도 규정에